# Frozen Lake – Proceso de Decisión de Markov (MDP)

En este notebook vamos a resolver el famoso entorno **Frozen Lake**, donde un agente debe cruzar un lago congelado desde el inicio hasta la meta.

El problema principal reside en que **el hielo es resbaladizo**: si el agente intenta moverse en una dirección, puede terminar desviándose a una dirección perpendicular con cierta probabilidad.

---

## Descripción del Entorno

El entorno es un **grid de 4×4** con las siguientes celdas:

| Símbolo | Significado | Recompensa |
|---------|-------------|------------|
| **S** | Start – posición inicial del agente | 0 |
| **F** | Frozen – camino seguro de hielo | 0 |
| **H** | Hole – agujero en el hielo (fin del juego) | 0 |
| **G** | Goal – meta del agente (fin del juego) | **+1** |

### Dinámica Estocástica

Al tomar una acción, el agente **no siempre se mueve a donde quiere**:

- **1/3 de probabilidad** → se mueve en la dirección deseada.
- **1/3 de probabilidad** → se desvía a uno de los lados perpendiculares.
- **1/3 de probabilidad** → se desvía al otro lado perpendicular.

Por ejemplo, si el agente elige ir al **Norte**, puede terminar yendo al Norte, al Este o al Oeste, cada uno con probabilidad 1/3.

---

## Representación del Grid

El lago se representa con una matriz 4×4:

```
S  F  F  F
F  H  F  H
F  F  F  H
H  F  F  G
```


In [10]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

frozen_lake_grid = np.array([
    ['S', 'F', 'F', 'F'],
    ['F', 'H', 'F', 'H'],
    ['F', 'F', 'F', 'H'],
    ['H', 'F', 'F', 'G']
])

print(frozen_lake_grid)

[['S' 'F' 'F' 'F']
 ['F' 'H' 'F' 'H']
 ['F' 'F' 'F' 'H']
 ['H' 'F' 'F' 'G']]


---

## Task 2.1 – Modelado del MDP

Para modelar el problema como un **Proceso de Decisión de Markov (MDP)**, necesitamos definir cuatro componentes clave:

1. **Estados** S: Los 16 estados posibles 0 - 15, numerados de izquierda a derecha y de arriba a abajo en el grid.
2. **Acciones** A: Cuatro acciones posibles → Norte (`N`), Este (`E`), Oeste (`W`), Sur (`S`).
3. **Función de Transición** T(s, a, s'): Probabilidad de llegar al estado $s'$ al tomar la acción $a$ desde el estado $s$. Captura la **estocasticidad** del hielo (1/3 por cada dirección posible).
4. **Función de Recompensa** R(s, a, s'): Recompensa obtenida al transitar de $s$ a $s'$ tomando la acción $a$. Solo es $+1$ si $s'$ es la meta (**G**); en cualquier otro caso es $0$.

### Funciones auxiliares

Antes de construir las matrices, definimos tres funciones auxiliares:

- `get_coord(state)` → Convierte un estado (0–15) a coordenadas `(fila, columna)`.
- `get_state(row, col)` → Convierte coordenadas `(fila, columna)` a estado (0–15).
- `try_move(row, col, direction_id)` → Calcula la nueva posición tras intentar moverse en una dirección, **respetando los límites del grid** (si choca con una pared, se queda en el mismo lugar).

### Diccionario de direcciones estocásticas

El diccionario `stochastic_directions` mapea cada acción a las **tres direcciones posibles** en las que el agente puede terminar:

| Acción | Dirección deseada | Desvíos posibles |
|--------|------------------|-----------------|
| Norte (0) | Norte | Este, Oeste |
| Este (1) | Este | Norte, Sur |
| Sur (2) | Sur | Este, Oeste |
| Oeste (3) | Oeste | Norte, Sur |


In [ ]:
n_states = 16
n_actions = 4

actions = ['N','E','W','S']

#Estados posibles en los que se peude estar (estado, accion, estado resultante)
T = np.zeros((n_states, n_actions, n_states)) #Transicion
R = np.zeros((n_states, n_actions, n_states)) #Recompensa

def get_coord(state):
    return state//4, state%4

def get_state(row, col):
    return row * 4 + col

def try_move(row, col, direction_id):
    if direction_id == 0: row = max(0, row - 1)    # Norte
    elif direction_id == 1: col = min(3, col + 1)  # Este
    elif direction_id == 2: row = min(3, row + 1)  # Sur
    elif direction_id == 3: col = max(0, col - 1)  # Oeste
    return row, col


stochastic_directions = {
    0: [0, 1, 3], # Norte (podemos ir al norte o resbalar a los lados)
    1: [1, 0, 2], # Este (podemos ir al este o resbalar a los lados)
    2: [2, 1, 3], # Sur (podemos ir al sur o resbalar a los lados)
    3: [3, 0, 2]  # Oeste (podemos ir al oeste o resbalar a los lados)
}
